# Description des Sources de Données

## A. Base de données ben_telecom
- Se connecter à la base de données
```sh
psql -d ben_telecom;
```

### 1. Catalogue d'offres (offers_catalog)

#### Table offers
```sh
\d+ offers_catalog.offers
```
- Contient les differents offres de communication téléphonique disponibles chez l'opérateur
- Colonnes

| Colonne   | Type de Donnée  | Description | 
|------------|-----------------------|-----------|
|id         | integer       |    Identifiant de l'offre |
|name       | varchar(50) |  Nom de l'offre |
|type       | varchar(10) |  Le type de l'offre
|price_xof  | integer  |  Le prix de l'offre en FCFA |
|start_date  | date  |  La date à laquelle l'offre a été créée |
|end_date  | date  | La date à laquelle l'offre a cessé d'exister (pour les offres qui ne sont plus en cours) |

- Clé primaire: `id`, auto-incrémentée
- Clé étrangère: Aucune
- Reférencée par
    - la table `offer_configurations`, qui réfère à sa clé primaire `id` = `offer_id`


#### Table billings

```sh
\d+ offers_catalog.offer_billings
```
- Contient les informations de facturation des différents offres disponibles chez l'opérateur
- Colonnes

| Colonne   | Type de Donnée  | Description | 
|------------|-----------------------|-----------|
|id         | integer       |    Identifiant de la facturation |
|xof_per_second  | numeric  |  Le prix en FCFA d'une seconde d'appel |
|xof_per_sms  | numeric  |  Le prix en FCFA d'un SMS envoyé |
|xof_per_mb  | numeric  |  Le prix en FCFA d'un  MB consommé |


- Clé primaire: `id`, auto-incrémentée
- Clé étrangère: Aucune
- Reférencée par
    - la table `offer_configurations`
        - qui réfère à sa clé primaire `id` = `billing_id`
        - Pour identifier l'offre utliser pour la configuration



#### Table offer_configurations

```sh
\d+ offers_catalog.offer_configurations
```
- Contient les informations de configuration des différents offres disponibles chez l'opérateur
- Colonnes

| Colonne   | Type de Donnée  | Description | 
|------------|-----------------------|-----------|
|id         | integer       |    Identifiant de la configuration d'offre |
|offer_id   | integer  |  L'identifiant de l'offre utlisée pour la configuration |
|value_mb  | numeric  |  La quantité de MB accordée par cette configuration d'offre |
|validity_days  | integer  |  Le délai de validité de la configuration d'offre en nombre de jours |
|is_limitless  | boolean  |  Détermine si l'offre configurée est une offre illimitée (ie toujours utlisable meme si les quotas des services accordes sont epuises) |
|start_date  | date  |  La date à laqauelle cette configuration d'offre a été créée |
|end_date  | date  | La date à laquelle cette configuration d'offre a cessé d'exister(pour les configuration d'offre qui ne sont plus en cours) |


- Clé primaire: `id`, auto-incrémentée
- Clé étrangères: 
    - `offer_id`
        - L'identifiant de l'offre utiliée pour la configuration
        - ie se refère à la colonne `id` de la table `offers`: `offers`(`id`)
    - `billing_id`
        - L'identifiant de la facturation utiliée pour la configuration
        - ie se refère à la colonne `id` de la table `offers_billing`: `offers_billing`(`id`)
- Reférencée par
    - Le répertoire des achats effectués
        - qui réfère à sa clé primaire `id` = `purchase_id`
        - Pour identifier la configuration d'offre achetée

### 2. CRM (crm) / Table customers

```sh
\d+ crm.customers
```

- Contient les informations sur les clients de l'opérateur.
- Colonnes

| Colonne   | Type de Donnée  | Description | 
|------------|-----------------------|-----------|
|id         | integer       |    Identifiant du client |
|city       | varchar(50) |  La ville de résidence du client |
|customer_type      | varchar(10) |  Le  type de client |
|gender      | char(1) |  Le sexe du client |
|customer_segment       | character varying(10) |  Le segment auquel appartient le client |
|birthdate      | date |  La date de naissance du client |
|registration_date      | date |  La date à laquelle le client s'est inscrit aupès de l'opérateur |
|status       | varchar(10) |  Le statut actuel du client |
|termination_date  | date |  La date à laquelle la relation de l'opérateur avec le client a pris fin |

- Clé primaire: `id`, auto-incrémentée
- Clé étrangère: Aucune
- Reférencée par
    - la table `subscriptions.subscriptions`
        - qui réfère à sa clé primaire `id` = `subscription_id`
        - Pour identifier le client à qui apartient la ligne téléphonique


### 3. Souscriptions (subscriptions) / Table subscriptions

```sh
\d+ subscriptions.subscriptions
```

- Contient les informations sur les différentes lignes téléphoiques des clients
- Colonnes

| Colonne   | Type de Donnée  | Description | 
|------------|-----------------------|-----------|
|id         | integer       |    Identifiant de la souscription |
|phone_number       | varchar(20) |  Le numéro de téléphone de la ligne téléphonique|
|customer_id         | integer       |    Identifiant du client qui s'enregistre sur cette ligne téléphonique |
|registration_date      | date |  La date à laquelle le client s'est s'enregistre sur cette ligne téléphonique  |
|status       | varchar(10) |  Le statut actuel du client |
|termination_date  | date |  La date à laquelle la ligne téléphonique a cesséde fonctionner |

- Clé primaire: `id`, auto-incrémentée
- Clé étrangères:
    - `customer_id`
    - L'identifiant du client à qui appartient la ligne téléphonique
    - ie se refère à la colonne `id` de la table `customers`: `customers`(`id`)
- Reférencée par
    - Le repertoire des achats
        - qui réfère à sa clé primaire `id` = `subscription_id`
        - pour identifier la  souscription / ligne téléphonique ayant acheté l'offre

## B. Les achats (purchases)
- Contient les différents achats effectués par les clients sur leurs lignes téléphoniques
    - Achats quotidiens dans des fichiers parquet

In [ ]:
import pandas as pd
import sys
from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.append(str(project_root / "config"))
from config import PURCHASES_FILE_PATH, ID_COL, PURCHASE_CHANNEL_COL, PAYMENT_METHOD_COL
print(PURCHASES_FILE_PATH)
purchases_df = pd.read_parquet(PURCHASES_FILE_PATH)
purchases_df

/home/starias/myprojects/telecom-offer-performance-analytics-platform/data/purchases


,id,subscription_id,offer_config_id,purchased_at,purchase_channel,payment_method
0,1,3751912,22,2020-05-20 15:10:28,WEB,AIRTIME
1,2,2907397,17,2020-05-21 18:20:14,DEALER,AIRTIME
2,3,5869941,30,2020-05-22 14:57:23,USSD,AIRTIME
3,4,3640231,28,2020-05-22 17:08:24,MOBILE_APP,MOBILE_MONEY
4,5,3230131,26,2020-05-23 01:51:54,USSD,AIRTIME
...,...,...,...,...,...,...
32413,32414,3810620,28,2020-08-29 23:56:30,WEB,AIRTIME
32414,32415,807485,12,2020-08-29 23:56:37,USSD,MOBILE_MONEY
32415,32416,3905232,18,2020-08-29 23:57:37,USSD,MOBILE_MONEY
32416,32417,5408,23,2020-08-29 23:58:40,USSD,AIRTIME


In [2]:
purchases_df.columns

Index(['id', 'subscription_id', 'offer_config_id', 'purchased_at',
       'purchase_channel', 'payment_method'],
      dtype='object')

In [3]:
purchases_df.dtypes

id                           int64
subscription_id              int64
offer_config_id              int64
purchased_at        datetime64[ns]
purchase_channel            object
payment_method              object
dtype: object

In [4]:
set(purchases_df[PURCHASE_CHANNEL_COL].values)

{'DEALER', 'MOBILE_APP', 'USSD', 'WEB'}

In [5]:
set(purchases_df[PAYMENT_METHOD_COL].values)

{'AIRTIME', 'MOBILE_MONEY'}

In [6]:
assert list(purchases_df[ID_COL].values) == list(range(1, len(purchases_df) + 1)) # Pour vérifier si c'est auto incémenté

- Contient les differents achats effectues par les clients
- Colonnes

| Colonne   | Type de Donnée  | Description | 
|------------|-----------------------|-----------|
|id         | entier      |    Identifiant de l'achat |
|subscription_id       | entier       |    La ligne téléphonique sur laquelle l'offre été achetée |
|offer_config_id       | entier      |    La configuration d'offre qui a étéachetée |
|purchased_at      | datetime |  La date et l'heure a laquelle l'offre a été acheté |
|purchase_channel      | string |  Le cannal par lequel l'achat a été effectué |
|payment_method      | string |  La méthode de paiement |

- Clé primaire: `id`, auto-incrémentée
- Clé étrangères:
    - `subscription_id` fait référence à `ben_telecom.subscriptions.subscriptions.id`
     - `offer_config_id` fait référence à `ben_telecom.offers_catalog.offer_configurations.id`
- Reférencée par
    - Le Système de Médiation d'Usage
        - qui réfère à sa clé primaire `id` = `purchase_id`
        - pour identifier l'offre utilisé pour le service téléphonique utilisé

In [ ]:
from kafka import KafkaConsumer
import json

topic = "TelecomUsage"
consumer = KafkaConsumer(
        topic,
        bootstrap_servers = "localhost:9092",
        auto_offset_reset="earliest",
        group_id = "consumer-test",
        consumer_timeout_ms = 5000
        # key_serializer=lambda key:str(key).encode("utf-8"),
  
        #value_deserializer=lambda value:json.loads(value.decode("utf-8"))
)
usage_df = pd.DataFrame([json.loads(message.value.decode("utf-8")) for message in consumer])

usage_df

KeyboardInterrupt: 

In [ ]:
usage_df

In [ ]:
usage_df.dtypes

purchase_id          int64
subscription_id      int64
timestamp           object
service_type        object
sms_count          float64
data_mb            float64
call_seconds       float64
dtype: object

## C. Système de Médiation d'Usage (Kafka)

### 0. Démarer le serveur Kafka (S'il n'est pas actif auparavant)

```sh
cd data
```

```sh
./kafka/bin/kafka-server-start.sh ./kafka/config/server.properties
```

### 1. Lister les topics
```sh
./kafka/bin/kafka-topics.sh  --bootstrap-server localhost:9092 --list
```

TelecomUsage est le seul topic

#### 2.  Décrie le topic Telecom Usage

```sh
./kafka/bin/kafka-topics.sh --describe --topic TelecomUsage --bootstrap-server localhost:9092
```







[Livrable: Dictionnaire de Données](../../docs/02)